In [1]:
!pip install kaggle

In [2]:
from google.colab import files
files.upload()   # upload kaggle.json

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"sar004","key":"aa9e8f04567dc9b38c80e85a8f4f06d7"}'}

In [3]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'brain-tumor-mri-dataset' dataset.
Path to dataset files: /kaggle/input/brain-tumor-mri-dataset


In [5]:
import os

print(os.listdir(path))

['Training', 'Testing']


In [6]:
for folder in os.listdir(path):
    print(folder, "→", os.listdir(os.path.join(path, folder))[:5])

Training → ['pituitary', 'notumor', 'meningioma', 'glioma']
Testing → ['pituitary', 'notumor', 'meningioma', 'glioma']


In [7]:
import os

train_dir = os.path.join(path, "Training")
test_dir = os.path.join(path, "Testing")

classes = ["glioma", "meningioma", "pituitary", "notumor"]

In [8]:
train_paths = []
train_labels = []

for cls in classes:
    cls_path = os.path.join(train_dir, cls)

    for img in os.listdir(cls_path):
        train_paths.append(os.path.join(cls_path, img))
        train_labels.append(cls)

In [9]:
print(len(train_paths))
print(train_labels[:5])

5600
['glioma', 'glioma', 'glioma', 'glioma', 'glioma']


In [10]:
test_paths = []
test_labels = []

for cls in classes:
    cls_path = os.path.join(test_dir, cls)

    for img in os.listdir(cls_path):
        test_paths.append(os.path.join(cls_path, img))
        test_labels.append(cls)

In [11]:
print(len(test_paths))
print(test_labels[:5])

1600
['glioma', 'glioma', 'glioma', 'glioma', 'glioma']


In [12]:
test_paths = []
test_labels = []

for cls in classes:
    cls_path = os.path.join(test_dir, cls)

    for img in os.listdir(cls_path):
        test_paths.append(os.path.join(cls_path, img))
        test_labels.append(cls)

In [13]:
print(len(test_paths))
print(test_labels[:5])

1600
['glioma', 'glioma', 'glioma', 'glioma', 'glioma']


In [14]:
import numpy as np

def create_labels(label_list):
    y_binary = []
    y_type = []

    for lbl in label_list:

        if lbl == "notumor":
            y_binary.append(0)
            y_type.append([0,0,0])

        elif lbl == "glioma":
            y_binary.append(1)
            y_type.append([1,0,0])

        elif lbl == "meningioma":
            y_binary.append(1)
            y_type.append([0,1,0])

        elif lbl == "pituitary":
            y_binary.append(1)
            y_type.append([0,0,1])

    return np.array(y_binary), np.array(y_type)

## 🎯 Multi-task Labels

Each image is assigned two labels:

1. Binary label:
   - 1 → Tumor present
   - 0 → No tumor

2. Tumor type:
   - Glioma → [1,0,0]
   - Meningioma → [0,1,0]
   - Pituitary → [0,0,1]

This allows the model to perform both detection and classification.


In [15]:
y_train_bin, y_train_type = create_labels(train_labels)
y_test_bin, y_test_type = create_labels(test_labels)

In [16]:
print(y_train_bin[:5])
print(y_train_type[:5])

[1 1 1 1 1]
[[1 0 0]
 [1 0 0]
 [1 0 0]
 [1 0 0]
 [1 0 0]]


In [17]:
import cv2

IMG_SIZE = 224

def load_images(paths):
    images = []

    for p in paths:
        img = cv2.imread(p)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        img = img / 255.0
        images.append(img)

    return np.array(images)

## ⚡ Custom Data Generator

A custom generator is used to load images in batches instead of loading the entire dataset into memory.

It also generates multi-task labels:
- Binary output (tumor detection)
- Type output (tumor classification)

In [18]:
BATCH_SIZE = 32
IMG_SIZE = 224

In [19]:
def data_generator(image_paths, labels, batch_size=BATCH_SIZE):

    while True:

        indices = np.arange(len(image_paths))
        np.random.shuffle(indices)

        image_paths_shuffled = np.array(image_paths)[indices]
        labels_shuffled = np.array(labels)[indices]

        for i in range(0, len(image_paths_shuffled), batch_size):

            batch_paths = image_paths_shuffled[i:i+batch_size]
            batch_labels = labels_shuffled[i:i+batch_size]

            X = []
            y_bin = []
            y_type = []

            for path, lbl in zip(batch_paths, batch_labels):

                img = cv2.imread(path)
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
                img = img / 255.0

                if lbl == "notumor":
                    y_bin.append(0)
                    y_type.append([0,0,0])
                else:
                    y_bin.append(1)

                    if lbl == "glioma":
                        y_type.append([1,0,0])
                    elif lbl == "meningioma":
                        y_type.append([0,1,0])
                    elif lbl == "pituitary":
                        y_type.append([0,0,1])

                X.append(img)

            yield np.array(X), {
                "binary_output": np.array(y_bin),
                "type_output": np.array(y_type)
            }

In [20]:
train_gen = data_generator(train_paths, train_labels)
test_gen = data_generator(test_paths, test_labels)

In [21]:
steps_per_epoch = len(train_paths) // BATCH_SIZE
validation_steps = len(test_paths) // BATCH_SIZE

print(steps_per_epoch, validation_steps)

175 50


## Multi-task Deep Learning Model

A pretrained EfficientNet model is used as a feature extractor.

The model has two outputs:
- Binary output for tumor detection
- Multi-class output for tumor type classification

This enables simultaneous learning of both tasks.

In [47]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Input
from tensorflow.keras.models import Model

# Input layer
input_layer = Input(shape=(224, 224, 3))

# Pretrained base
base_model = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_tensor=input_layer
)

# Freeze base model (Phase 1)
for layer in base_model.layers:
    layer.trainable = False

# Feature extraction
x = base_model.output
x = GlobalAveragePooling2D()(x)

# 🔹 Output 1: Binary (Tumor)
binary_output = Dense(1, activation="sigmoid", name="binary_output")(x)

# 🔹 Output 2: Type (3 classes)
type_output = Dense(3, activation="softmax", name="type_output")(x)

# Final model
model = Model(inputs=input_layer, outputs=[binary_output, type_output])

model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_4         │ (None, 224, 224,  │          0 │ input_layer_2[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization_2     │ (None, 224, 224,  │          7 │ rescaling_4[0][0] │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_5         │ (None, 224, 224,  │          0 │ normalization_2[… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 225, 225,  │          0 │ rescaling_5[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │        864 │ stem_conv_pad[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 112, 112,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 112, 112,  │        288 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 112, 112,  │        128 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 112, 112,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 32)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 32)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 8)   │        264 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 32)  │        288 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 112, 112,  │          0 │ block1a_activati… │
│ (Multiply)          │ 32)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 112, 112,  │        512 │ block1a_se_excit

 Total params: 4,054,695 (15.47 MB)

 Trainable params: 5,124 (20.02 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

 Model Compilation

The model is compiled with two loss functions:
- Binary crossentropy for tumor detection
- Categorical crossentropy for tumor classification

Both tasks are optimized simultaneously.

In [23]:
 #Unfreeze last layers
for layer in base_model.layers[-50:]:
    layer.trainable = True

In [24]:
import tensorflow as tf
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),

    loss={
        "binary_output": "binary_crossentropy",
        "type_output": "categorical_crossentropy"
    },

    loss_weights={
        "binary_output": 1.0,
        "type_output": 2.0   # boosting the weight/importance
    },

    metrics={
        "binary_output": "accuracy",
        "type_output": "accuracy"
    }
)

In [25]:
history_finetune = model.fit(
    train_gen,
    steps_per_epoch=steps_per_epoch,
    validation_data=test_gen,
    validation_steps=validation_steps,
    epochs=10
)

Epoch 1/10
175/175 ━━━━━━━━━━━━━━━━━━━━ 111s 417ms/step - binary_output_accuracy: 0.7138 - binary_output_loss: 0.5970 - loss: 2.2556 - type_output_accuracy: 0.3545 - type_output_loss: 0.8293 - val_binary_output_accuracy: 0.7500 - val_binary_output_loss: 0.6098 - val_loss: 2.2590 - val_type_output_accuracy: 0.2500 - val_type_output_loss: 0.8246
Epoch 2/10
175/175 ━━━━━━━━━━━━━━━━━━━━ 31s 175ms/step - binary_output_accuracy: 0.7588 - binary_output_loss: 0.5393 - loss: 2.1829 - type_output_accuracy: 0.3627 - type_output_loss: 0.8218 - val_binary_output_accuracy: 0.7500 - val_binary_output_loss: 0.5708 - val_loss: 2.2297 - val_type_output_accuracy: 0.4794 - val_type_output_loss: 0.8295
Epoch 3/10
175/175 ━━━━━━━━━━━━━━━━━━━━ 29s 164ms/step - binary_output_accuracy: 0.7704 - binary_output_loss: 0.5263 - loss: 2.1677 - type_output_accuracy: 0.3571 - type_output_loss: 0.8207 - val_binary_output_accuracy: 0.8094 - val_binary_output_loss: 0.5807 - val_loss: 2.2111 - val_type_output_accuracy: 0.

##  Custom Loss for Multi-task Learning

To improve tumor type classification, a masked loss is used.

- Type loss is applied only to tumor images
- Non-tumor samples are ignored for type prediction

This ensures that the model learns meaningful class distinctions without confusion.

In [29]:
import tensorflow as tf

def masked_type_loss(y_true, y_pred):

    # detect valid samples (tumor wale)
    mask = tf.reduce_sum(y_true, axis=1)  # notumor → 0, tumor → 1

    # normal categorical loss
    loss = tf.keras.losses.categorical_crossentropy(y_true, y_pred)

    # apply mask (ignore notumor)
    loss = loss * mask

    return tf.reduce_mean(loss)

## Optimized Data Generator

A custom data generator is implemented to efficiently load images in batches.

Enhancements include:
- On-the-fly image loading (memory efficient)
- Data augmentation (flip & rotation)
- Multi-task labels (binary + type)
- Stable output format for seamless model training

This improves generalization and avoids memory overflow issues.

In [48]:
import numpy as np
import cv2

BATCH_SIZE = 16   #  reduce for better learning
IMG_SIZE = 224

def data_generator_v2(image_paths, labels, batch_size=BATCH_SIZE):

    while True:

        indices = np.arange(len(image_paths))
        np.random.shuffle(indices)

        image_paths_shuffled = np.array(image_paths)[indices]
        labels_shuffled = np.array(labels)[indices]

        for i in range(0, len(image_paths_shuffled), batch_size):

            batch_paths = image_paths_shuffled[i:i+batch_size]
            batch_labels = labels_shuffled[i:i+batch_size]

            X = []
            y_bin = []
            y_type = []

            for path, lbl in zip(batch_paths, batch_labels):

                img = cv2.imread(path)
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

                # 🔥 AUGMENTATION
                if np.random.rand() < 0.5:
                    img = cv2.flip(img, 1)

                if np.random.rand() < 0.3:
                    angle = np.random.randint(-15, 15)
                    h, w = img.shape[:2]
                    M = cv2.getRotationMatrix2D((w//2, h//2), angle, 1)
                    img = cv2.warpAffine(img, M, (w, h))

                img = img / 255.0

                # 🔹 Binary
                if lbl == "notumor":
                    y_bin.append(0)
                    y_type.append([0,0,0])

                else:
                    y_bin.append(1)

                    if lbl == "glioma":
                        y_type.append([1,0,0])
                    elif lbl == "meningioma":
                        y_type.append([0,1,0])
                    elif lbl == "pituitary":
                        y_type.append([0,0,1])

                X.append(img)

            yield np.array(X), (
                np.array(y_bin),
                np.array(y_type)
            )

In [49]:
train_gen = data_generator_v2(train_paths, train_labels)
test_gen = data_generator_v2(test_paths, test_labels)

In [51]:
base_model.trainable = True

In [52]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),

    loss=[
        "binary_crossentropy",
        masked_type_loss
    ],

    loss_weights=[1.0, 2.0],  # 🔥 boost type

    metrics=["accuracy", "accuracy"]
)

In [53]:
history_masked = model.fit(
    train_gen,
    steps_per_epoch=steps_per_epoch,
    validation_data=test_gen,
    validation_steps=validation_steps,
    epochs=20
)

Epoch 1/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 42s 119ms/step - binary_output_accuracy: 0.7504 - binary_output_loss: 0.6234 - loss: 2.3576 - type_output_accuracy: 0.5036 - type_output_loss: 0.8671 - val_binary_output_accuracy: 0.7337 - val_binary_output_loss: 0.6100 - val_loss: 2.2573 - val_type_output_accuracy: 0.5175 - val_type_output_loss: 0.8236
Epoch 2/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 13s 74ms/step - binary_output_accuracy: 0.7496 - binary_output_loss: 0.5902 - loss: 2.2700 - type_output_accuracy: 0.4961 - type_output_loss: 0.8399 - val_binary_output_accuracy: 0.7688 - val_binary_output_loss: 0.5669 - val_loss: 2.2760 - val_type_output_accuracy: 0.4800 - val_type_output_loss: 0.8546
Epoch 3/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 13s 72ms/step - binary_output_accuracy: 0.7457 - binary_output_loss: 0.5785 - loss: 2.2281 - type_output_accuracy: 0.4932 - type_output_loss: 0.8248 - val_binary_output_accuracy: 0.7500 - val_binary_output_loss: 0.5700 - val_loss: 2.2207 - val_type_output_accuracy: 0.496